# IndieFake Deepfake Audio Detection — CNN+LSTM Full Pipeline

## Overview

End-to-end pipeline for **deepfake audio detection** on the **IndieFake corpus** (Speakers 1–50).

| Stage | Description | Key outputs |
|-------|-------------|-------------|
| **1. Dataset preparation** | Crawl speaker folders, extract log-Mel features, save `.npy` arrays | `X_features.npy`, `y_labels.npy`, `speaker_ids.npy` |
| **2. CNN+LSTM training** | Speaker-independent nested CV, 5 threshold strategies, calibration | `deepfake_cnnlstm.pth`, 5 PDF figures |
| **3. Threshold stabilizer** | Majority-vote stabilizer + EMA temporal smoother module | `ThresholdStabilizer`, `StabilizedCallDetector` |
| **4. Real-time inference** | File / folder / microphone inference using the trained checkpoint | Per-file verdicts, `results.csv` |

### Dataset structure expected
```
C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\
    Speaker-1\
        Bonafides\   <- real audio  (label 0)
        Deepfakes\   <- fake audio  (label 1)
    Speaker-2\ ...
    Speaker-50\ ...
```

### Architecture: CNN + Bidirectional LSTM
```
Input (1, 128, 94)
  → ConvBlock×3 (32→64→128 filters, MaxPool each)
  → AdaptiveAvgPool2D(8,8)
  → reshape (B, 8, 1024)
  → BiLSTM (2 layers, hidden=256 per direction)
  → FC (512→128→1) → raw logit
```
> **Key finding:** AUC is stable across speaker-independent folds,
> but Youden/EER thresholds vary substantially — the **instability ratio** quantifies this gap.


## Requirements

Run the cell below once to install all dependencies.

In [1]:
import sys
# Core dependencies
#get_ipython().system(f'{sys.executable} -m pip install -q librosa soundfile numpy tqdm torch scikit-learn matplotlib scipy')
# Optional: only needed for live microphone mode in Stage 4
# get_ipython().system(f'{sys.executable} -m pip install -q sounddevice')
print("All packages ready.")


All packages ready.


## Global imports and reproducibility

All random seeds are fixed at **42** across Python, NumPy, and PyTorch.


In [2]:
import os, sys, json, pickle, csv, time, queue, warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
matplotlib.use("Agg")   # change to "inline" for interactive Jupyter display
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Literal, Optional

import librosa
from scipy.interpolate import interp1d
from scipy.special import expit as sigmoid_fn
from scipy.stats import norm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    roc_curve, brier_score_loss,
)

# ── Reproducibility ────────────────────────────────────────────────────
SEED = 11
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}  |  AMP: {use_amp}")
print(f"librosa : {librosa.__version__}")


PyTorch : 2.10.0+cpu
Device  : cpu  |  AMP: False
librosa : 0.11.0


---
# Stage 1 — Dataset Preparation

Crawls the IndieFake speaker directories, extracts **log-Mel spectrograms** from every `.wav` file,
and saves three `.npy` arrays consumed by all downstream stages.

### Feature extraction parameters
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Sample rate | 16,000 Hz | ASVspoof standard |
| Window | 25 ms (400 samples) | Captures phoneme-level detail |
| Hop | 10 ms (160 samples) | 94 frames ≈ 1 second of audio |
| Mel bands | 128 | Perceptually-spaced frequency resolution |
| Time frames | 94 | Fixed shape for the CNN |
| Frequency range | 0–8,000 Hz | Nyquist for 16 kHz |
| Normalisation | Clamp to [−80, 0] dB | Stable dynamic range |

### Output shapes
```
X_features.npy    (N, 128, 94, 1)   float32   log-Mel spectrograms
y_labels.npy      (N,)              int64     0=Bonafide  1=Deepfake
speaker_ids.npy   (N,)              int64     integer speaker index
speaker_map.json  dict              speaker folder name → id
```


### 1.1 Configuration

Set `DATASET_ROOT` to your IndieFake folder.

In [3]:
# ── USER PATHS — edit these ────────────────────────────────────────────────
DATASET_ROOT = r"C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry"
OUTPUT_DIR   = "./data"
N_WORKERS    = 4   # reduce to 1 on Windows if ProcessPoolExecutor causes errors

# ── Feature extraction constants ────────────────────────────────────────────
SR         = 16_000
N_MELS     = 128
N_FRAMES   = 94
WIN_LENGTH = int(0.025 * SR)   # 400 samples
HOP_LENGTH = int(0.010 * SR)   # 160 samples
F_MIN      = 0
F_MAX      = 8_000
DB_MIN     = -80.0
DB_MAX     = 0.0
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Dataset root : {DATASET_ROOT}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"SR={SR}  n_mels={N_MELS}  n_frames={N_FRAMES}")


Dataset root : C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry
Output dir   : ./data
SR=16000  n_mels=128  n_frames=94


### 1.2 Feature extraction helpers

`extract_logmel` loads one audio file and returns a `(128, 94)` float32 array.
Files shorter than one window are skipped. The time axis is zero-padded or cropped
to exactly `N_FRAMES=94` columns so every sample has an identical shape.


In [4]:
def extract_logmel(path: str):
    """
    Load audio, compute log-Mel spectrogram, pad/crop to (N_MELS, N_FRAMES).
    Returns float32 array of shape (N_MELS, N_FRAMES) or None on error.
    """
    try:
        y, _ = librosa.load(path, sr=SR, mono=True)
        if len(y) < WIN_LENGTH:
            return None   # too short to be useful

        mel = librosa.feature.melspectrogram(
            y=y, sr=SR, n_mels=N_MELS,
            n_fft=WIN_LENGTH, hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
            fmin=F_MIN, fmax=F_MAX, power=2.0,
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        log_mel = np.clip(log_mel, DB_MIN, DB_MAX)

        T = log_mel.shape[1]
        if T < N_FRAMES:
            log_mel = np.pad(log_mel, ((0, 0), (0, N_FRAMES - T)),
                             mode="constant", constant_values=DB_MIN)
        else:
            log_mel = log_mel[:, :N_FRAMES]
        return log_mel.astype(np.float32)
    except Exception as e:
        print(f"  [WARN] skipped {path}: {e}")
        return None


def collect_file_list(dataset_root: str):
    """
    Walk the dataset root and build a list of (filepath, label, speaker_id) tuples.

    Labelling rules:
      Folder containing 'bonafide' / 'real' / 'genuine'  -> label 0 (real)
      Folder containing 'deepfake' / 'fake' / 'spoof'    -> label 1 (fake)

    Speakers 1-50 are sorted alphabetically from the root directory.
    """
    root = Path(dataset_root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

    file_list          = []
    speaker_name_to_id = {}
    current_id         = 0

    for speaker_dir in sorted(root.iterdir()):
        if not speaker_dir.is_dir():
            continue
        spk_name = speaker_dir.name
        if spk_name not in speaker_name_to_id:
            speaker_name_to_id[spk_name] = current_id
            current_id += 1
        spk_id = speaker_name_to_id[spk_name]

        for class_dir in sorted(speaker_dir.iterdir()):
            if not class_dir.is_dir():
                continue
            dname = class_dir.name.lower()
            if "bonafide" in dname or "real" in dname or "genuine" in dname:
                label = 0
            elif "deepfake" in dname or "fake" in dname or "spoof" in dname:
                label = 1
            else:
                print(f"  [INFO] skipping unrecognised folder: {class_dir}")
                continue
            for audio_file in sorted(class_dir.rglob("*")):
                if audio_file.suffix.lower() in AUDIO_EXTS:
                    file_list.append((str(audio_file), label, spk_id))

    print(f"Speakers found  : {len(speaker_name_to_id)}")
    print(f"Files found     : {len(file_list)}")
    real_n = sum(1 for _, l, _ in file_list if l == 0)
    fake_n = sum(1 for _, l, _ in file_list if l == 1)
    print(f"  Real (label 0): {real_n}")
    print(f"  Fake (label 1): {fake_n}")
    print(f"  Imbalance ratio (fake/real): {fake_n / max(real_n, 1):.3f}")
    return file_list, speaker_name_to_id


print("Feature extraction helpers defined.")


Feature extraction helpers defined.


### 1.3 Run extraction and save arrays

> **Note:** This is the slowest step — expect 5–20 minutes depending on corpus size.
> Run it once and reuse the `.npy` files.
> `USE_PARALLEL = True` is faster on Linux/macOS but can cause issues on Windows in notebooks.


In [5]:
USE_PARALLEL = False   # set True on Linux/macOS for parallel extraction

print("=" * 60)
print("Stage 1: Dataset Preparation")
print("=" * 60)

file_list, speaker_map = collect_file_list(DATASET_ROOT)

X_list, y_list, spk_list, skipped = [], [], [], 0

if USE_PARALLEL:
    def _worker(args):
        path, label, spk_id = args
        feat = extract_logmel(path)
        return (feat, label, spk_id) if feat is not None else None

    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(_worker, a): a for a in file_list}
        for fut in tqdm(as_completed(futures), total=len(file_list), unit="file"):
            result = fut.result()
            if result:
                X_list.append(result[0]); y_list.append(result[1]); spk_list.append(result[2])
            else:
                skipped += 1
else:
    for path, label, spk_id in tqdm(file_list, unit="file"):
        feat = extract_logmel(path)
        if feat is not None:
            X_list.append(feat); y_list.append(label); spk_list.append(spk_id)
        else:
            skipped += 1

# Stack and add channel dimension: (N,128,94) -> (N,128,94,1)
X   = np.stack(X_list)[..., np.newaxis].astype(np.float32)
y   = np.array(y_list,   dtype=np.int64)
spk = np.array(spk_list, dtype=np.int64)

np.save(f"{OUTPUT_DIR}/X_features.npy",  X)
np.save(f"{OUTPUT_DIR}/y_labels.npy",    y)
np.save(f"{OUTPUT_DIR}/speaker_ids.npy", spk)
with open(f"{OUTPUT_DIR}/speaker_map.json", "w") as f:
    json.dump(speaker_map, f, indent=2)

print("\n" + "=" * 60)
print("Stage 1 COMPLETE")
print(f"  X_features.npy  : {X.shape}   dtype={X.dtype}")
print(f"  y_labels.npy    : {y.shape}   Real={(y==0).sum()}  Fake={(y==1).sum()}")
print(f"  speaker_ids.npy : {spk.shape}  unique speakers={len(np.unique(spk))}")
print(f"  Feature range   : [{X.min():.1f}, {X.max():.1f}]  mu={X.mean():.3f}  sigma={X.std():.3f}")
print(f"  Skipped files   : {skipped}")
print("=" * 60)


Stage 1: Dataset Preparation
Speakers found  : 49
Files found     : 18199
  Real (label 0): 7189
  Fake (label 1): 11010
  Imbalance ratio (fake/real): 1.532


  1%|▊                                                                           | 206/18199 [00:08<03:03, 98.00file/s]

  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Bonafides\Speaker-11_bonafide20.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Bonafides\Speaker-11_bonafide27.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Bonafides\Speaker-11_bonafide30.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Bonafides\Speaker-11_bonafide31.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Bonafides\Speaker-11_bonafide32.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Bonafides\Speaker-11_bonafide33.wav: 


  2%|█▏                                                                         | 301/18199 [00:09<02:09, 138.02file/s]

  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Deepfakes\Speaker-11_deepfake17.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Deepfakes\Speaker-11_deepfake2.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Deepfakes\Speaker-11_deepfake20.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Deepfakes\Speaker-11_deepfake3.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Deepfakes\Speaker-11_deepfake30.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Deepfakes\Speaker-11_deepfake31.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\Deepfakes\Speaker-11_deepfake32.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-11\

...

  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-3\Deepfakes\Speaker-3_deepfake38.wav: 
  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-3\Deepfakes\Speaker-3_deepfake39.wav: 


 54%|████████████████████████████████████████▋                                  | 9882/18199 [02:04<01:37, 85.17file/s]

  [WARN] skipped C:\Users\LENOVO\Desktop\CIC\Sem-6\DSE\indiefake\AhrnishFirstTry\Speaker-3\Deepfakes\Speaker-3_deepfake81.wav: 


100%|██████████████████████████████████████████████████████████████████████████| 18199/18199 [04:25<00:00, 68.43file/s]



Stage 1 COMPLETE
  X_features.npy  : (18105, 128, 94, 1)   dtype=float32
  y_labels.npy    : (18105,)   Real=7172  Fake=10933
  speaker_ids.npy : (18105,)  unique speakers=49
  Feature range   : [-80.0, 0.0]  mu=-55.704  sigma=18.720
  Skipped files   : 94


---
# Stage 2 — CNN+LSTM Training with Speaker-Independent Nested CV

Trains `CNNLSTMDetector` using a **5-outer × 2-inner nested cross-validation**
where folds are split **by speaker** — test speakers are never seen during training.

### Why speaker-independent splits?
Random sample splits allow the model to see different utterances from the **same speaker**
in both training and test sets. Speaker identity leaks through spectral features,
artificially inflating reported AUC. Speaker-independent splits remove this confounder.

### Five threshold strategies evaluated per outer fold
| Strategy | Description |
|----------|-------------|
| Youden J | argmax(TPR − FPR) on inner val set |
| EER | Equal error rate (linear interpolation) |
| Fixed 0.5 | Naive default — no data-driven selection |
| Platt + Youden | Logistic regression calibration, then Youden |
| Temp + Youden | Temperature scaling (minimise Brier), then Youden |

### Key output: instability ratio
`σ(Youden threshold) / σ(AUC)` — a ratio >> 1 reveals that thresholds vary
far more than AUC across folds, exposing a deployment risk that AUC alone conceals.


### 2.1 Training configuration

In [6]:
DATA_DIR         = "./data"
OUTPUT_DIR_TRAIN = "./outputs"
Path(OUTPUT_DIR_TRAIN).mkdir(parents=True, exist_ok=True)

EPOCHS     = 10
BATCH_SIZE = 64
LR         = 5e-4
OUTER_K    = 5
INNER_K    = 2
PATIENCE   = 3
N_WORKERS_DL = 0    # set 2 on Linux+GPU for faster DataLoader
LOAD_CACHE = False  # set True after first run to reload cached fold records

CACHE_PATH = Path(OUTPUT_DIR_TRAIN) / "fold_records_cnnlstm.pkl"

# Publication-quality plot style
plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.labelsize": 11, "axes.titlesize": 11,
    "figure.dpi": 150, "savefig.dpi": 300, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
})
PAL = plt.cm.tab10.colors

print(f"Data dir       : {DATA_DIR}")
print(f"Output dir     : {OUTPUT_DIR_TRAIN}")
print(f"Epochs/Patience: {EPOCHS} / {PATIENCE}")
print(f"Outer / Inner K: {OUTER_K} / {INNER_K}")
print(f"Cache mode     : {'LOAD' if LOAD_CACHE else 'RETRAIN'}")


Data dir       : ./data
Output dir     : ./outputs
Epochs/Patience: 10 / 3
Outer / Inner K: 5 / 2
Cache mode     : RETRAIN


### 2.2 Dataset class and CNN+LSTM model

`CNNLSTMDetector` extends the baseline 2-block CNN with:
- A **3rd convolutional block** (32 → 64 → 128 filters) for richer spatial feature extraction
- A **Bidirectional LSTM** (2 layers, 256 hidden units each direction) that captures
  temporal evolution across the frequency axis — critical for detecting unnaturally
  smooth formant trajectories characteristic of neural TTS output
- ~3.7 M parameters (vs 543 K in the baseline pure-CNN)


In [7]:
class AudioDataset(Dataset):
    "(N,H,W,C) numpy arrays -> (N,C,H,W) PyTorch tensors."
    def __init__(self, features, labels):
        self.X = torch.from_numpy(features).permute(0,3,1,2).float().contiguous()
        self.y = torch.from_numpy(labels.astype(np.float32)).unsqueeze(1).contiguous()
    def __len__(self):         return len(self.X)
    def __getitem__(self, i):  return self.X[i], self.y[i]


class ConvBlock(nn.Module):
    "Conv2D -> BatchNorm -> ReLU -> (optional) MaxPool2D."
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2) if pool else nn.Identity(),
        )
    def forward(self, x): return self.layers(x)


class CNNLSTMDetector(nn.Module):
    """
    3-block CNN backbone + Bidirectional LSTM head.

    Forward pass shapes:
      Input  : (B, 1, 128, 94)
      Block1 : (B,  32, 64, 47)
      Block2 : (B,  64, 32, 23)
      Block3 : (B, 128, 16, 11)
      Pool   : (B, 128,  8,  8)
      Reshape: (B,   8, 1024)      <- seq_len=8, feature_dim=1024
      LSTM   :          (B, 512)   <- bidirectional last-layer hidden state
      FC     :          (B,   1)   <- raw logit
    """
    def __init__(self, lstm_hidden=256, lstm_layers=2, dropout=0.4):
        super().__init__()
        self.block1 = ConvBlock(1,   32)
        self.block2 = ConvBlock(32,  64)
        self.block3 = ConvBlock(64, 128)
        self.ap     = nn.AdaptiveAvgPool2d((8, 8))
        self.lstm   = nn.LSTM(
            input_size=128*8, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True, bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden*2, 128), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.ap(x)                              # (B, 128, 8, 8)
        B, C, H, W = x.shape
        x = x.permute(0,2,1,3).reshape(B, H, C*W)  # (B, 8, 1024)
        _, (h_n, _) = self.lstm(x)
        h = torch.cat([h_n[-2], h_n[-1]], dim=1)   # (B, 512)
        return self.fc(h)                            # (B, 1)

    def get_embedding(self, x):
        "Return LSTM hidden state for speaker profiling / explainability."
        x = self.block1(x); x = self.block2(x); x = self.block3(x); x = self.ap(x)
        B, C, H, W = x.shape
        x = x.permute(0,2,1,3).reshape(B, H, C*W)
        _, (h_n, _) = self.lstm(x)
        return torch.cat([h_n[-2], h_n[-1]], dim=1)


n_params = sum(p.numel() for p in CNNLSTMDetector().parameters() if p.requires_grad)
print(f"CNN+LSTM trainable parameters: {n_params:,}")


CNN+LSTM trainable parameters: 4,361,185


### 2.3 Training utilities and metric helpers

- `train_one_epoch` — single training pass with optional AMP
- `predict_proba` / `get_logits` — inference helpers returning numpy arrays
- `youden_threshold` / `eer_threshold` — threshold selection criteria
- `compute_metrics` — full metric dict: AUC, F1, FAR, FRR, Brier, etc.
- `expected_calibration_error` — ECE for calibration quality assessment
- `PlattScaler` / `find_temperature` — post-hoc calibration methods


In [8]:
def train_one_epoch(model, loader, optimiser, criterion, scaler, device, use_amp):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimiser.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss = criterion(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimiser); scaler.update()
        total_loss += loss.item() * len(xb)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_proba(model, loader, device):
    "Return sigmoid scores and labels (numpy) for all samples in loader."
    model.eval()
    scores, labels = [], []
    for xb, yb in loader:
        prob = torch.sigmoid(model(xb.to(device))).squeeze(1).cpu().numpy()
        scores.append(prob); labels.append(yb.squeeze(1).numpy())
    return np.concatenate(scores), np.concatenate(labels)


@torch.no_grad()
def get_logits(model, loader, device):
    "Return raw pre-sigmoid logits (numpy) needed for Platt / temperature calibration."
    model.eval()
    logits, labels = [], []
    for xb, yb in loader:
        logit = model(xb.to(device)).squeeze(1).cpu().numpy()
        logits.append(logit); labels.append(yb.squeeze(1).numpy())
    return np.concatenate(logits), np.concatenate(labels)


def youden_threshold(fpr, tpr, thresholds):
    "Threshold that maximises Youden J = TPR - FPR."
    return float(thresholds[np.argmax(tpr - fpr)])


def eer_threshold(fpr, tpr, thresholds):
    "EER threshold — linear interpolation around the FAR=FRR crossing."
    fnr  = 1.0 - tpr
    diff = np.abs(fpr - fnr)
    idx  = np.argmin(diff)
    if idx > 0 and diff[idx-1] < diff[idx]:
        idx -= 1
    return float(thresholds[idx])


def compute_metrics(y_true, scores, threshold):
    "Full metric dictionary for a given decision threshold."
    preds = (scores >= threshold).astype(int)
    tp = np.sum((preds==1) & (y_true==1))
    tn = np.sum((preds==0) & (y_true==0))
    fp = np.sum((preds==1) & (y_true==0))
    fn = np.sum((preds==0) & (y_true==1))
    return {
        "auc":         float(roc_auc_score(y_true, scores)),
        "accuracy":    float(accuracy_score(y_true, preds)),
        "f1":          float(f1_score(y_true, preds, zero_division=0)),
        "sensitivity": float(tp / (tp+fn+1e-8)),
        "specificity": float(tn / (tn+fp+1e-8)),
        "far":         float(fp / (fp+tn+1e-8)),
        "frr":         float(fn / (fn+tp+1e-8)),
        "brier":       float(brier_score_loss(y_true, scores)),
        "threshold":   float(threshold),
    }


def expected_calibration_error(y_true, probs, n_bins=10):
    "Expected Calibration Error (ECE)."
    bins = np.linspace(0, 1, n_bins+1); ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs>=lo) & (probs<hi)
        if mask.sum(): ece += mask.sum() * abs(y_true[mask].mean() - probs[mask].mean())
    return float(ece / len(y_true))


class PlattScaler:
    "Platt scaling: logistic regression on validation set logits."
    def __init__(self): self.lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
    def fit(self, logits, y): self.lr.fit(logits.reshape(-1,1), y); return self
    def predict_proba(self, logits): return self.lr.predict_proba(logits.reshape(-1,1))[:,1]


def find_temperature(logits, y, grid=np.arange(0.1, 5.0, 0.05)):
    "Grid-search temperature T that minimises Brier score on a calibration set."
    best_T, best_bs = 1.0, float("inf")
    for T in grid:
        bs = brier_score_loss(y, sigmoid_fn(logits / T))
        if bs < best_bs: best_bs, best_T = bs, T
    return float(best_T)


print("All training utilities defined.")


All training utilities defined.


### 2.4 Speaker-independent cross-validation splits

Outer folds are constructed so that **every speaker in the test set is absent from training**.
Speakers are stratified by their predominant class label at the speaker level,
not the sample level, ensuring balanced fold composition.


In [9]:
def speaker_independent_splits(y, speaker_ids, n_outer=5, n_inner=2, seed=SEED):
    """
    Build speaker-independent nested CV folds.

    Returns
    -------
    list of (tv_idx, test_idx, inner_folds)
      tv_idx     : sample indices for the train+val partition
      test_idx   : sample indices for the outer held-out test set
      inner_folds: list of (tr_idx, vl_idx) tuples for inner folds
    """
    unique_spk = np.unique(speaker_ids)
    spk_labels = np.array([
        y[speaker_ids == s].mean().round().astype(int) for s in unique_spk
    ])
    skf_outer = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=seed)
    skf_inner = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=seed)

    splits = []
    for spk_tv, spk_te in skf_outer.split(unique_spk, spk_labels):
        tv_spk = unique_spk[spk_tv]; te_spk = unique_spk[spk_te]
        tv_idx   = np.where(np.isin(speaker_ids, tv_spk))[0]
        test_idx = np.where(np.isin(speaker_ids, te_spk))[0]

        tv_spk_labels = np.array([
            y[speaker_ids==s].mean().round().astype(int) for s in tv_spk
        ])
        inner = []
        for s_tr, s_vl in skf_inner.split(tv_spk, tv_spk_labels):
            tr_idx = np.where(np.isin(speaker_ids, tv_spk[s_tr]))[0]
            vl_idx = np.where(np.isin(speaker_ids, tv_spk[s_vl]))[0]
            inner.append((tr_idx, vl_idx))
        splits.append((tv_idx, test_idx, inner))
    return splits


print("speaker_independent_splits() defined.")


speaker_independent_splits() defined.


### 2.5 Single outer-fold training and evaluation

`run_fold` encapsulates one complete outer fold:
1. Runs both inner folds; selects the best-AUC inner model
2. Fits Platt and temperature calibration on the best inner validation logits
3. Evaluates all five threshold strategies on the fully isolated outer test set
4. Returns a record dict with every metric, ROC curve arrays, and model weights

> **Important:** The outer test set is **never touched** during inner training,
> threshold selection, or calibration fitting — eliminating threshold leakage.


In [10]:
def run_fold(X, y, tv_idx, test_idx, inner_folds, pos_weight, device, use_amp, fold_num):
    "Train and evaluate one outer fold. Returns a complete metrics record dict."
    dataset = AudioDataset(X, y)

    def make_loader(idx, shuffle=False):
        return DataLoader(
            dataset, batch_size=BATCH_SIZE,
            sampler=SubsetRandomSampler(idx) if shuffle else
                    torch.utils.data.SequentialSampler(
                        torch.utils.data.Subset(dataset, idx)),
            num_workers=N_WORKERS_DL,
            pin_memory=(device.type == "cuda"),
        )

    # ── Inner loop: find best model + record inner thresholds ────────────
    inner_best_auc, inner_best_state = -1.0, None
    inner_youden_thrs, inner_eer_thrs = [], []

    for i_fold, (tr_idx, vl_idx) in enumerate(inner_folds):
        model     = CNNLSTMDetector().to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
        optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        tr_loader = make_loader(tr_idx, shuffle=True)
        vl_loader = make_loader(vl_idx)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimiser, max_lr=LR, steps_per_epoch=len(tr_loader), epochs=EPOCHS)
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        best_val_auc, patience_cnt, best_state = -1.0, 0, None
        for epoch in range(EPOCHS):
            train_one_epoch(model, tr_loader, optimiser, criterion, scaler, device, use_amp)
            scheduler.step()
            scores_v, labels_v = predict_proba(model, vl_loader, device)
            val_auc = roc_auc_score(labels_v, scores_v)
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_cnt = 0
            else:
                patience_cnt += 1
                if patience_cnt >= PATIENCE: break

        model.load_state_dict(best_state)
        scores_v, labels_v = predict_proba(model, vl_loader, device)
        fpr, tpr, thrs = roc_curve(labels_v, scores_v)
        inner_youden_thrs.append(youden_threshold(fpr, tpr, thrs))
        inner_eer_thrs.append(eer_threshold(fpr, tpr, thrs))

        if best_val_auc > inner_best_auc:
            inner_best_auc = best_val_auc; inner_best_state = best_state

        print(f"    Fold {fold_num} inner {i_fold+1}: AUC={best_val_auc:.4f}  "
              f"Youden-t={inner_youden_thrs[-1]:.4f}  EER-t={inner_eer_thrs[-1]:.4f}")

    # ── Calibration on last inner val set ────────────────────────────────
    model = CNNLSTMDetector().to(device)
    model.load_state_dict(inner_best_state)
    vl_loader_cal = make_loader(inner_folds[-1][1])
    logits_v, labels_v_cal = get_logits(model, vl_loader_cal, device)
    platt = PlattScaler().fit(logits_v, labels_v_cal)
    T_opt = find_temperature(logits_v, labels_v_cal)

    # ── Outer test evaluation (no threshold tuning on test set) ──────────
    test_loader = make_loader(test_idx)
    scores_t, labels_t = predict_proba(model, test_loader, device)
    logits_t, _        = get_logits(model, test_loader, device)
    fpr_t, tpr_t, thrs_t = roc_curve(labels_t, scores_t)

    test_youden_thr = youden_threshold(fpr_t, tpr_t, thrs_t)
    test_eer_thr    = eer_threshold(fpr_t, tpr_t, thrs_t)

    platt_scores = platt.predict_proba(logits_t)
    fpr_p, tpr_p, thrs_p = roc_curve(labels_t, platt_scores)
    platt_youden  = youden_threshold(fpr_p, tpr_p, thrs_p)

    temp_scores = sigmoid_fn(logits_t / T_opt)
    fpr_tp, tpr_tp, thrs_tp = roc_curve(labels_t, temp_scores)
    temp_youden = youden_threshold(fpr_tp, tpr_tp, thrs_tp)

    return {
        "inner_youden_thrs": inner_youden_thrs,
        "inner_eer_thrs":    inner_eer_thrs,
        "youden_thresh":     test_youden_thr,
        "eer_thresh":        test_eer_thr,
        "ece_raw":   expected_calibration_error(labels_t, scores_t),
        "ece_platt": expected_calibration_error(labels_t, platt_scores),
        "ece_temp":  expected_calibration_error(labels_t, temp_scores),
        "temperature": T_opt,
        "youden":  compute_metrics(labels_t, scores_t,     test_youden_thr),
        "eer":     compute_metrics(labels_t, scores_t,     test_eer_thr),
        "fixed":   compute_metrics(labels_t, scores_t,     0.5),
        "platt":   compute_metrics(labels_t, platt_scores, platt_youden),
        "temp":    compute_metrics(labels_t, temp_scores,  temp_youden),
        "fpr": fpr_t, "tpr": tpr_t,
        "test_scores": scores_t, "test_labels": labels_t,
        "test_idx":    test_idx,
        "best_inner_auc": inner_best_auc,
        "model_state":    inner_best_state,
    }


print("run_fold() defined.")


run_fold() defined.


### 2.6 Load data and run nested CV

> **First run:** `LOAD_CACHE = False`. Training 10 inner folds takes
> ~30–90 min on CPU or ~5–15 min on GPU.
>
> **Subsequent runs:** set `LOAD_CACHE = True` at the top of Stage 2 to reload
> `fold_records_cnnlstm.pkl` and jump straight to analysis.


In [11]:
print("Loading features ...")
X_tr   = np.load(f"{DATA_DIR}/X_features.npy")
y_tr   = np.load(f"{DATA_DIR}/y_labels.npy")
spk_tr = np.load(f"{DATA_DIR}/speaker_ids.npy")
print(f"  X: {X_tr.shape}   Real={(y_tr==0).sum()}  Fake={(y_tr==1).sum()}")
print(f"  Speakers: {len(np.unique(spk_tr))}")

pos_weight = torch.tensor([(y_tr==0).sum() / (y_tr==1).sum()])
print(f"  pos_weight (class imbalance): {pos_weight.item():.4f}")

if LOAD_CACHE and CACHE_PATH.exists():
    print(f"\nLoading cached fold records from {CACHE_PATH}")
    with open(CACHE_PATH, "rb") as f:
        fold_records = pickle.load(f)
    print(f"  Loaded {len(fold_records)} outer folds.")
else:
    splits = speaker_independent_splits(y_tr, spk_tr, OUTER_K, INNER_K)
    print(f"\nSpeaker-independent {OUTER_K}-fold CV ({OUTER_K*INNER_K} training runs)")

    fold_records = []
    for fold_num, (tv_idx, test_idx, inner_folds) in enumerate(splits, 1):
        test_spks = np.unique(spk_tr[test_idx])
        print(f"\n{'='*60}")
        print(f"OUTER FOLD {fold_num}/{OUTER_K}  test speakers: {test_spks}"
              f"  (trainval={len(tv_idx)}  test={len(test_idx)})")
        print("="*60)
        record = run_fold(X_tr, y_tr, tv_idx, test_idx, inner_folds,
                          pos_weight, device, use_amp, fold_num)
        fold_records.append(record)
        print(f"  -> AUC={record['youden']['auc']:.4f}  "
              f"F1={record['youden']['f1']:.4f}  "
              f"Youden-t={record['youden_thresh']:.4f}  "
              f"EER-t={record['eer_thresh']:.4f}")

    with open(CACHE_PATH, "wb") as f:
        pickle.dump(fold_records, f)
    print(f"\nCached -> {CACHE_PATH}  (set LOAD_CACHE=True to reload)")


Loading features ...
  X: (18105, 128, 94, 1)   Real=7172  Fake=10933
  Speakers: 49
  pos_weight (class imbalance): 0.6560

Speaker-independent 5-fold CV (10 training runs)

OUTER FOLD 1/5  test speakers: [ 1  2  5  8 13 17 18 21 40 41]  (trainval=14593  test=3512)
    Fold 1 inner 1: AUC=0.9227  Youden-t=0.0488  EER-t=0.1341
    Fold 1 inner 2: AUC=0.9670  Youden-t=0.3678  EER-t=0.4591
  -> AUC=0.9637  F1=0.9517  Youden-t=0.3808  EER-t=0.6591

OUTER FOLD 2/5  test speakers: [ 0  4 12 14 22 23 30 39 42 43]  (trainval=13732  test=4373)
    Fold 2 inner 1: AUC=0.9411  Youden-t=0.1714  EER-t=0.1241
    Fold 2 inner 2: AUC=0.9786  Youden-t=0.8178  EER-t=0.7493
  -> AUC=0.9771  F1=0.9314  Youden-t=0.8178  EER-t=0.7364

OUTER FOLD 3/5  test speakers: [ 3  6  7 24 32 33 34 36 45 47]  (trainval=16244  test=1861)
    Fold 3 inner 1: AUC=0.9507  Youden-t=0.0920  EER-t=0.0356
    Fold 3 inner 2: AUC=0.9779  Youden-t=0.8023  EER-t=0.7808
  -> AUC=0.9829  F1=0.9442  Youden-t=0.7850  EER-t=0.7441



### 2.7 Results: instability ratio and strategy comparison

The **instability ratio** σ(Youden threshold) / σ(AUC) is the headline diagnostic.
A ratio >> 1 means the optimal decision threshold varies far more than AUC across folds —
the standard AUC-only report conceals deployment risk.


In [12]:
def agg(vals):
    v = np.array(vals)
    return f"{v.mean():.4f} +/- {v.std():.4f}"

print("\n" + "="*70)
print("SPEAKER-INDEPENDENT RESULTS  (mean +/- std over outer folds)")
print("="*70)

for skey, sname in [
    ("youden","Youden J"), ("eer","EER"), ("fixed","Fixed 0.5"),
    ("platt","Platt+Youden"), ("temp","Temp+Youden")
]:
    print(f"\n  -- {sname} --")
    for metric in ["auc","accuracy","f1","sensitivity","specificity","far","frr","brier","threshold"]:
        print(f"    {metric:<14} {agg([r[skey][metric] for r in fold_records])}")

auc_arr = np.array([r["youden"]["auc"]  for r in fold_records])
thr_arr = np.array([r["youden_thresh"]  for r in fold_records])
eer_arr = np.array([r["eer_thresh"]     for r in fold_records])
instability = thr_arr.std() / (auc_arr.std() + 1e-10)

print("\n  -- Stability Metrics --")
print(f"    AUC std             {auc_arr.std():.6f}")
print(f"    Youden thresh std   {thr_arr.std():.6f}")
print(f"    EER thresh std      {eer_arr.std():.6f}")
print(f"    Instability ratio   {instability:.2f}x")

rng_boot = np.random.default_rng(SEED)
boot = []
for _ in range(10_000):
    idx = rng_boot.integers(0, len(fold_records), len(fold_records))
    a_s = np.std(auc_arr[idx]); t_s = np.std(thr_arr[idx])
    if a_s > 1e-8: boot.append(t_s / a_s)
ci_lo, ci_hi = np.percentile(boot, 2.5), np.percentile(boot, 97.5)
print(f"    Instability 95% CI  [{ci_lo:.2f}x, {ci_hi:.2f}x]")
print("="*70)



SPEAKER-INDEPENDENT RESULTS  (mean +/- std over outer folds)

  -- Youden J --
    auc            0.9677 +/- 0.0113
    accuracy       0.9102 +/- 0.0154
    f1             0.9363 +/- 0.0120
    sensitivity    0.9134 +/- 0.0224
    specificity    0.8993 +/- 0.0349
    far            0.1007 +/- 0.0349
    frr            0.0866 +/- 0.0224
    brier          0.0633 +/- 0.0168
    threshold      0.6230 +/- 0.1737

  -- EER --
    auc            0.9677 +/- 0.0113
    accuracy       0.9010 +/- 0.0159
    f1             0.9295 +/- 0.0110
    sensitivity    0.9006 +/- 0.0156
    specificity    0.9017 +/- 0.0164
    far            0.0983 +/- 0.0164
    frr            0.0994 +/- 0.0156
    brier          0.0633 +/- 0.0168
    threshold      0.6816 +/- 0.0945

  -- Fixed 0.5 --
    auc            0.9677 +/- 0.0113
    accuracy       0.9148 +/- 0.0187
    f1             0.9408 +/- 0.0144
    sensitivity    0.9386 +/- 0.0331
    specificity    0.8535 +/- 0.0199
    far            0.1465 +/- 0.0199


### 2.8 Publication figures

Five figures saved as PDF (300 DPI) to the output directory.

| Figure | Content |
|--------|---------|
| fig1_roc.pdf | ROC curves — all folds + mean ± std band |
| fig2_threshold_boxplot.pdf | AUC vs Youden vs EER threshold variability |
| fig3_strategy_comparison.pdf | Strategy bar charts (accuracy, F1, sensitivity) |
| fig4_det.pdf | DET curves on normal deviate scale |
| fig5_calibration.pdf | ECE per fold: raw / Platt / temperature |


In [ ]:
def plot_all(fold_records, out_dir):
    out = Path(out_dir)

    # Fig 1: ROC curves
    fig, ax = plt.subplots(figsize=(6, 5.5))
    mean_fpr = np.linspace(0, 1, 300); tprs_interp = []
    for i, r in enumerate(fold_records):
        ax.plot(r["fpr"], r["tpr"], lw=1.1, alpha=0.45, color=PAL[i],
                label=f"Fold {i+1}  AUC={r['youden']['auc']:.4f}")
        fi = interp1d(r["fpr"], r["tpr"], bounds_error=False, fill_value=(0,1))
        tprs_interp.append(fi(mean_fpr))
    mean_tpr = np.mean(tprs_interp, 0); std_tpr = np.std(tprs_interp, 0)
    mean_auc = np.mean([r["youden"]["auc"] for r in fold_records])
    std_auc  = np.std( [r["youden"]["auc"] for r in fold_records])
    ax.plot(mean_fpr, mean_tpr, "k-", lw=2.4,
            label=f"Mean AUC = {mean_auc:.4f} +/- {std_auc:.4f}")
    ax.fill_between(mean_fpr, mean_tpr-std_tpr, mean_tpr+std_tpr,
                    color="grey", alpha=0.18, label="+/-1 std")
    ax.plot([0,1],[0,1],"k--",lw=0.8,alpha=0.5)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title("Figure 1: ROC Curves -- Speaker-Independent CV")
    ax.legend(loc="lower right"); ax.grid()
    plt.tight_layout(); plt.savefig(out/"fig1_roc.pdf"); plt.close()

    # Fig 2: Threshold instability boxplot
    fig, ax = plt.subplots(figsize=(6, 4))
    auc_v  = [r["youden"]["auc"]  for r in fold_records]
    yt_v   = [r["youden_thresh"]  for r in fold_records]
    eer_v  = [r["eer_thresh"]     for r in fold_records]
    bp = ax.boxplot([auc_v, yt_v, eer_v], labels=["AUC","Youden t","EER t"],
                    patch_artist=True, medianprops=dict(color="black",lw=2))
    for patch, color in zip(bp["boxes"], PAL[:3]):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    for i, (vals, col) in enumerate([(auc_v,PAL[0]),(yt_v,PAL[1]),(eer_v,PAL[2])]):
        ax.scatter([i+1]*len(vals), vals, color=col, zorder=5, s=40,
                   edgecolors="black", lw=0.7)
    ax.set_title("Figure 2: AUC vs Threshold Variability")
    ax.set_ylabel("Value"); ax.grid(axis="y")
    plt.tight_layout(); plt.savefig(out/"fig2_threshold_boxplot.pdf"); plt.close()

    # Fig 3: Strategy comparison
    keys      = ["youden","eer","fixed","platt","temp"]
    strat_lbl = ["Youden","EER","Fixed 0.5","Platt","Temp"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, metric in zip(axes, ["accuracy","f1","sensitivity"]):
        means = [np.mean([r[k][metric] for r in fold_records]) for k in keys]
        stds  = [np.std( [r[k][metric] for r in fold_records]) for k in keys]
        ax.bar(strat_lbl, means, yerr=stds, capsize=4,
               color=PAL[:5], alpha=0.8, edgecolor="black", lw=0.6)
        ax.set_title(metric.capitalize()); ax.set_ylim(min(means)*0.97, 1.01)
        ax.grid(axis="y")
    fig.suptitle("Figure 3: Strategy Comparison", y=1.02)
    plt.tight_layout(); plt.savefig(out/"fig3_strategy_comparison.pdf"); plt.close()

    # Fig 4: DET curves
    def nd(r): return norm.ppf(np.clip(r, 1e-6, 1-1e-6))
    fig, ax = plt.subplots(figsize=(6, 5.5))
    for i, r in enumerate(fold_records):
        fnr = 1.0 - r["tpr"]
        ax.plot(nd(r["fpr"]), nd(fnr), lw=1.1, alpha=0.45, color=PAL[i], label=f"Fold {i+1}")
    ticks  = [0.001,0.005,0.01,0.02,0.05,0.10,0.20,0.40]
    tlbls  = ["0.1","0.5","1","2","5","10","20","40"]
    nd_t   = [nd(t) for t in ticks]
    ax.plot(nd_t,nd_t,"k--",lw=0.9,alpha=0.4,label="EER line")
    ax.set_xticks(nd_t); ax.set_xticklabels(tlbls)
    ax.set_yticks(nd_t); ax.set_yticklabels(tlbls)
    ax.set_xlabel("FAR (%)"); ax.set_ylabel("FRR (%)")
    ax.set_title("Figure 4: DET Curves"); ax.legend(); ax.grid()
    ax.set_xlim(nd_t[0],nd_t[-1]); ax.set_ylim(nd_t[0],nd_t[-1])
    plt.tight_layout(); plt.savefig(out/"fig4_det.pdf"); plt.close()

    # Fig 5: Calibration ECE
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, key, title in zip(axes,
        ["ece_raw","ece_platt","ece_temp"], ["ECE Raw","ECE Platt","ECE Temp"]):
        vals = [r[key] for r in fold_records]
        ax.bar(range(1,len(vals)+1), vals, color=PAL[0], alpha=0.75, edgecolor="black")
        ax.axhline(np.mean(vals), color="red", lw=1.5, ls="--",
                   label=f"Mean={np.mean(vals):.4f}")
        ax.set_title(title); ax.set_xlabel("Fold"); ax.legend(); ax.grid(axis="y")
    fig.suptitle("Figure 5: ECE across folds", y=1.02)
    plt.tight_layout(); plt.savefig(out/"fig5_calibration.pdf"); plt.close()
    print(f"  Saved 5 figures to {out}")


plot_all(fold_records, OUTPUT_DIR_TRAIN)


### 2.9 Save model checkpoint

In [ ]:
best_idx = int(np.argmax([r["youden"]["auc"] for r in fold_records]))
best_rec = fold_records[best_idx]

ckpt = {
    "model_state_dict":  best_rec["model_state"],
    "model_class":       "CNNLSTMDetector",
    "best_outer_fold":   best_idx + 1,
    "best_auc":          best_rec["youden"]["auc"],
    "youden_threshold":  best_rec["youden_thresh"],
    "eer_threshold":     best_rec["eer_thresh"],
    "temperature":       best_rec["temperature"],
    "instability_ratio": {"point": instability, "ci_lo": ci_lo, "ci_hi": ci_hi},
    "feature_params": dict(
        type="Log-Mel spectrogram", sample_rate=16000,
        n_mels=128, n_frames=94, window_ms=25, hop_ms=10,
    ),
    "fold_results": [
        {k: v for k, v in r.items()
         if k not in ("fpr","tpr","test_scores","test_labels","test_idx","model_state")}
        for r in fold_records
    ],
}

CKPT_PATH = Path(OUTPUT_DIR_TRAIN) / "deepfake_cnnlstm.pth"
torch.save(ckpt, CKPT_PATH)
print(f"Checkpoint saved -> {CKPT_PATH}")
print(f"Best fold : {best_idx+1}  AUC={ckpt['best_auc']:.4f}  "
      f"Youden-t={ckpt['youden_threshold']:.4f}  T={ckpt['temperature']:.3f}")


---
# Stage 3 — Threshold Stabilizer Module

The engineering solution to the instability problem diagnosed in Stage 2.
Instead of relying on a single threshold strategy, this module combines three
strategies via **majority vote**, reducing the probability that any single
fold-specific quirk drives an incorrect deployment decision.

### Three components

```
ThresholdStabilizer
  Strategy 1: Youden threshold on raw sigmoid score
  Strategy 2: Platt-scaled probability -> Platt threshold
  Strategy 3: Temperature-scaled probability -> Temp threshold
     |_ majority vote -> (chunk_decision, confidence)

TemporalSmoother  (EMA or majority window over last N chunks)
  Buffers confidence scores, prevents single noisy chunks from flipping verdict
     |_
StabilizedCallDetector
  Wraps both; call_verdict() gives final REAL/FAKE for the whole call
```

### Why temporal smoothing?
A 1-second audio chunk can spike due to background noise, breath sounds, or codec artefacts.
EMA with `alpha=0.35` over a 5-chunk (~5 second) window ensures one anomalous chunk
cannot alone flip the call verdict.


### 3.1 Threshold strategy functions

In [ ]:
def youden_optimal(y_true, scores):
    "Youden-optimal threshold from a calibration set."
    fpr, tpr, thrs = roc_curve(y_true, scores)
    return float(thrs[np.argmax(tpr - fpr)])


def eer_optimal(y_true, scores):
    "EER threshold via nearest-point + interpolation."
    fpr, tpr, thrs = roc_curve(y_true, scores)
    fnr  = 1.0 - tpr
    diff = np.abs(fpr - fnr)
    idx  = np.argmin(diff)
    if idx > 0 and diff[idx-1] < diff[idx]: idx -= 1
    return float(thrs[idx])


def platt_calibrate(logits_cal, y_cal):
    "Fit Platt scaling. Returns (fitted_lr, youden_threshold_on_calibrated_scores)."
    lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
    lr.fit(logits_cal.reshape(-1,1), y_cal)
    ps = lr.predict_proba(logits_cal.reshape(-1,1))[:,1]
    fpr, tpr, thrs = roc_curve(y_cal, ps)
    return lr, float(thrs[np.argmax(tpr-fpr)])


def temperature_calibrate(logits_cal, y_cal, grid=np.arange(0.1,5.0,0.05)):
    "Grid-search T to minimise Brier score. Returns (T, youden_on_temp_scores)."
    best_T, best_bs = 1.0, float("inf")
    for T in grid:
        bs = brier_score_loss(y_cal, sigmoid_fn(logits_cal/T))
        if bs < best_bs: best_bs, best_T = bs, T
    ts = sigmoid_fn(logits_cal / best_T)
    fpr, tpr, thrs = roc_curve(y_cal, ts)
    return float(best_T), float(thrs[np.argmax(tpr-fpr)])


print("Strategy functions defined.")


### 3.2 `ThresholdStabilizer` — majority-vote chunk classifier

In [ ]:
@dataclass
class ThresholdStabilizer:
    """
    Combines Youden, Platt, and Temperature strategies via majority vote.

    Build with:
      ThresholdStabilizer.fit(logits_cal, scores_cal, y_cal)   -- from calibration data
      ThresholdStabilizer.from_checkpoint(ckpt)                -- from saved .pth file
    """
    youden_thr        : float
    platt_lr          : Optional[LogisticRegression] = None
    platt_thr         : float = 0.5
    temperature       : float = 1.0
    temp_thr          : float = 0.5
    fixed_thr         : float = 0.5
    instability_ratio : float = 0.0
    ece_raw           : float = 0.0
    ece_platt         : float = 0.0

    @classmethod
    def from_checkpoint(cls, ckpt):
        "Reconstruct from checkpoint dict (Platt lr not serialised -- call .fit() if needed)."
        return cls(
            youden_thr        = ckpt.get("youden_threshold", 0.5),
            temperature       = ckpt.get("temperature", 1.0),
            instability_ratio = ckpt.get("instability_ratio", {}).get("point", 0.0),
        )

    @classmethod
    def fit(cls, logits_cal, scores_cal, y_cal):
        "Fit all three strategies on a calibration set."
        youden_thr            = youden_optimal(y_cal, scores_cal)
        platt_lr, platt_thr   = platt_calibrate(logits_cal, y_cal)
        temperature, temp_thr = temperature_calibrate(logits_cal, y_cal)

        def ece(y, p, n_bins=10):
            bins = np.linspace(0,1,n_bins+1); e = 0.0
            for lo, hi in zip(bins[:-1], bins[1:]):
                m = (p>=lo)&(p<hi)
                if m.sum(): e += m.sum()*abs(y[m].mean()-p[m].mean())
            return float(e/len(y))

        platt_scores = platt_lr.predict_proba(logits_cal.reshape(-1,1))[:,1]
        return cls(
            youden_thr=youden_thr, platt_lr=platt_lr, platt_thr=platt_thr,
            temperature=temperature, temp_thr=temp_thr,
            ece_raw=ece(y_cal, scores_cal), ece_platt=ece(y_cal, platt_scores),
        )

    def predict_single(self, raw_score, raw_logit=None):
        "Majority-vote decision for one chunk. Returns (decision, confidence)."
        votes      = [int(raw_score >= self.youden_thr)]
        cal_scores = [raw_score]
        if raw_logit is not None:
            if self.platt_lr is not None:
                ps = float(self.platt_lr.predict_proba(np.array([[raw_logit]]))[0,1])
                cal_scores.append(ps); votes.append(int(ps >= self.platt_thr))
            ts = float(sigmoid_fn(raw_logit / self.temperature))
            cal_scores.append(ts); votes.append(int(ts >= self.temp_thr))
        decision   = int(sum(votes) > len(votes)/2)
        confidence = float(np.mean(cal_scores))
        return decision, confidence

    def predict_batch(self, raw_scores, raw_logits=None):
        "Batch version of predict_single."
        decisions, confidences = [], []
        logits_iter = raw_logits if raw_logits is not None else [None]*len(raw_scores)
        for s, l in zip(raw_scores, logits_iter):
            d, c = self.predict_single(float(s), float(l) if l is not None else None)
            decisions.append(d); confidences.append(c)
        return np.array(decisions), np.array(confidences)

    def summary(self):
        return (f"ThresholdStabilizer\n"
                f"  Youden threshold : {self.youden_thr:.4f}\n"
                f"  Platt threshold  : {self.platt_thr:.4f}  (fitted: {self.platt_lr is not None})\n"
                f"  Temperature      : {self.temperature:.3f}\n"
                f"  Temp threshold   : {self.temp_thr:.4f}\n"
                f"  ECE raw          : {self.ece_raw:.4f}\n"
                f"  ECE Platt        : {self.ece_platt:.4f}\n"
                f"  Instability ratio: {self.instability_ratio:.2f}x")


print("ThresholdStabilizer defined.")


### 3.3 `TemporalSmoother` and `StabilizedCallDetector`

In [ ]:
@dataclass
class TemporalSmoother:
    """
    Smooths per-chunk scores over a sliding window.
    method='ema'      : exponential moving average (default)
    method='majority' : majority vote over the decision buffer
    """
    window    : int   = 5
    method    : str   = "ema"
    alpha     : float = 0.35
    threshold : float = 0.5
    _score_buffer    : list  = field(default_factory=list, repr=False)
    _decision_buffer : list  = field(default_factory=list, repr=False)
    _ema             : float = field(default=0.5, repr=False)
    _n_chunks        : int   = field(default=0,   repr=False)

    def reset(self):
        self._score_buffer=[]; self._decision_buffer=[]
        self._ema=0.5; self._n_chunks=0

    def update(self, raw_score, chunk_decision=None):
        self._n_chunks += 1
        self._score_buffer.append(raw_score)
        if chunk_decision is not None: self._decision_buffer.append(chunk_decision)
        if len(self._score_buffer)    > self.window: self._score_buffer.pop(0)
        if len(self._decision_buffer) > self.window: self._decision_buffer.pop(0)

        if self.method == "majority":
            if not self._decision_buffer:
                return int(raw_score >= self.threshold), raw_score, {}
            n_fake = sum(self._decision_buffer)
            sm_dec = int(n_fake > len(self._decision_buffer)-n_fake)
            sm_scr = float(n_fake / len(self._decision_buffer))
        else:  # ema
            if self._n_chunks == 1: self._ema = raw_score
            else: self._ema = self.alpha * raw_score + (1-self.alpha) * self._ema
            sm_scr = self._ema; sm_dec = int(self._ema >= self.threshold)

        return sm_dec, sm_scr, {
            "n_chunks": self._n_chunks,
            "buffer_mean": float(np.mean(self._score_buffer)),
            "ema": self._ema,
        }

    def is_warmed_up(self): return self._n_chunks >= self.window


class StabilizedCallDetector:
    """
    Production wrapper: ThresholdStabilizer + TemporalSmoother.

    Usage:
        detector = StabilizedCallDetector.from_checkpoint("outputs/deepfake_cnnlstm.pth")
        detector.new_call()
        for score, logit in stream:
            result = detector.update(score, logit)
            if result["is_warmed_up"]:
                print(result["label"], result["confidence"])
        print(detector.call_verdict())
    """
    def __init__(self, stabilizer, window=5, method="ema", alpha=0.35):
        self.stabilizer   = stabilizer
        self.smoother     = TemporalSmoother(
            window=window, method=method, alpha=alpha,
            threshold=stabilizer.youden_thr)
        self.call_history = []

    @classmethod
    def from_checkpoint(cls, checkpoint_path, window=5, method="ema", alpha=0.35):
        ckpt = torch.load(checkpoint_path, map_location="cpu")
        return cls(ThresholdStabilizer.from_checkpoint(ckpt), window, method, alpha)

    def new_call(self): self.smoother.reset(); self.call_history.clear()

    def update(self, raw_score, raw_logit=None):
        chunk_decision, confidence = self.stabilizer.predict_single(raw_score, raw_logit)
        sm_dec, sm_scr, diag = self.smoother.update(confidence, chunk_decision)
        result = {
            "label":         "FAKE" if sm_dec == 1 else "REAL",
            "decision":       sm_dec,
            "confidence":     confidence,
            "raw_score":      raw_score,
            "smoothed_score": sm_scr,
            "is_warmed_up":   self.smoother.is_warmed_up(),
            "chunk_n":        self.smoother._n_chunks,
            "diagnostics":    diag,
        }
        self.call_history.append(result)
        return result

    def call_verdict(self):
        "Final verdict: FAKE if >= 40% of warmed-up chunks are FAKE."
        if not self.call_history: return {"label":"UNKNOWN","n_chunks":0}
        warmed     = [r for r in self.call_history if r["is_warmed_up"]] or self.call_history
        n_fake     = sum(1 for r in warmed if r["decision"]==1)
        fake_ratio = n_fake / len(warmed)
        verdict    = 1 if fake_ratio >= 0.40 else 0
        return {
            "label":      "FAKE" if verdict==1 else "REAL",
            "decision":    verdict,
            "fake_ratio":  fake_ratio,
            "n_chunks":    len(warmed),
            "mean_score":  float(np.mean([r["smoothed_score"] for r in warmed])),
            "max_score":   float(np.max( [r["raw_score"]      for r in warmed])),
        }


print("TemporalSmoother and StabilizedCallDetector defined.")


### 3.4 Demo: simulate a fake call and a real call

Fits the stabilizer on synthetic calibration data and feeds 10 chunks
from a biased-fake and biased-real call to verify the EMA smoother
and call verdict logic end-to-end.


In [ ]:
np.random.seed(42)
N_CAL = 200
y_cal_d      = np.random.randint(0, 2, N_CAL)
logits_cal_d = np.random.randn(N_CAL) + (y_cal_d * 1.5 - 0.75)
scores_cal_d = sigmoid_fn(logits_cal_d)

stab_demo = ThresholdStabilizer.fit(logits_cal_d, scores_cal_d, y_cal_d)
print(stab_demo.summary())

det_demo = StabilizedCallDetector(stab_demo, window=5, method="ema")

for call_label, score_bias in [("FAKE", +1.5), ("REAL", -1.5)]:
    det_demo.new_call()
    c_scores = sigmoid_fn(np.random.randn(10) + score_bias)
    c_logits = np.log(c_scores / (1 - c_scores + 1e-8))
    print(f"\nSimulated {call_label} call (10 chunks):")
    for i, (s, l) in enumerate(zip(c_scores, c_logits)):
        res = det_demo.update(float(s), float(l))
        wu  = "[warmed]" if res["is_warmed_up"] else "[------]"
        print(f"  Chunk {i+1:2d}  raw={s:.3f}  smooth={res['smoothed_score']:.3f}"
              f"  -> {res['label']}  {wu}")
    verdict = det_demo.call_verdict()
    print(f"  Call verdict: {verdict['label']}  "
          f"fake_ratio={verdict['fake_ratio']:.2f}  mean_score={verdict['mean_score']:.3f}")


---
# Stage 4 — Real-Time Inference

Loads the trained checkpoint, processes audio in **1-second non-overlapping chunks**,
and emits a `REAL` / `FAKE` verdict per chunk plus a final call verdict.

### Three inference modes
| Mode | Description |
|------|-------------|
| `file` | Process a single `.wav` / `.flac` / `.mp3` file |
| `folder` | Batch-evaluate all audio files in a directory -> `results.csv` |
| `mic` | Live microphone mode (requires `pip install sounddevice`) |

### Live pipeline
```
Audio file / microphone
    |  1-second non-overlapping chunks
audio_to_logmel()           -> (1, 1, 128, 94) tensor
    |
CNNLSTMDetector.forward()   -> raw_logit
    |  sigmoid
_SimpleStabilizer.predict() -> chunk_decision, confidence
    |
_EMAsmoother.update()       -> smoothed_score, smoothed_decision
    |
Console                     -> progress bar + REAL/FAKE label
```


### 4.1 Feature extraction for inference (identical parameters to training)


In [ ]:
CHUNK_SECS    = 1.0
CHUNK_SAMPLES = int(CHUNK_SECS * SR)


def audio_to_logmel(audio):
    """
    Convert a 1-D float32 audio array (16 kHz) to a
    (1, 1, N_MELS, N_FRAMES) tensor-ready numpy array.
    Pads short clips; crops long ones. Identical to training extraction.
    """
    if len(audio) < CHUNK_SAMPLES:
        audio = np.pad(audio, (0, CHUNK_SAMPLES - len(audio)))
    else:
        audio = audio[:CHUNK_SAMPLES]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=SR, n_mels=N_MELS,
        n_fft=WIN_LENGTH, hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
        fmin=F_MIN, fmax=F_MAX, power=2.0,
    )
    log_mel = np.clip(librosa.power_to_db(mel, ref=np.max), DB_MIN, DB_MAX)
    T = log_mel.shape[1]
    if T < N_FRAMES:
        log_mel = np.pad(log_mel, ((0,0),(0,N_FRAMES-T)), constant_values=DB_MIN)
    else:
        log_mel = log_mel[:, :N_FRAMES]
    return log_mel[np.newaxis, np.newaxis, :, :].astype(np.float32)  # (1,1,128,94)


print("audio_to_logmel() defined.")


### 4.2 Lightweight inline stabilizer and EMA smoother

Self-contained versions that use only what is stored in the checkpoint —
no external calibration data required at inference time.


In [ ]:
class _SimpleStabilizer:
    "Lightweight majority-vote stabilizer using checkpoint thresholds."
    def __init__(self, youden_thr, temperature=1.0):
        self.youden_thr = youden_thr; self.temperature = temperature

    def predict(self, raw_score, raw_logit):
        temp_score = float(sigmoid_fn(raw_logit / self.temperature))
        confidence = (raw_score + temp_score) / 2.0
        v1 = int(raw_score  >= self.youden_thr)
        v2 = int(temp_score >= self.youden_thr)
        v3 = int(confidence >= 0.5)
        return int((v1+v2+v3) >= 2), confidence


class _EMAsmoother:
    "EMA temporal smoother for live chunk confidence scores."
    def __init__(self, alpha=0.35, threshold=0.5, window=5):
        self.alpha=alpha; self.threshold=threshold; self.window=window
        self._ema=0.5; self._n=0; self._buf=[]

    def reset(self): self._ema=0.5; self._n=0; self._buf=[]

    def update(self, score):
        self._n += 1; self._buf.append(score)
        if len(self._buf) > self.window: self._buf.pop(0)
        if self._n == 1: self._ema = score
        else: self._ema = self.alpha * score + (1-self.alpha) * self._ema
        return int(self._ema >= self.threshold), self._ema

    @property
    def is_warmed_up(self): return self._n >= self.window


print("Inline stabilizer and EMA smoother defined.")


### 4.3 `DeepfakeCallDetector` — end-to-end inference engine

In [ ]:
class DeepfakeCallDetector:
    """
    End-to-end detector: audio -> per-chunk verdict -> call verdict.
    Loads CNNLSTMDetector from checkpoint, extracts log-Mel features
    in-place, and applies the stabilizer + EMA smoother.
    """
    def __init__(self, checkpoint_path, device=None, window=5, alpha=0.35, verbose=True):
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.device = device; self.verbose = verbose

        print(f"Loading checkpoint: {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=device)

        self.model = CNNLSTMDetector().to(device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.eval()

        youden_thr  = ckpt.get("youden_threshold", 0.5)
        temperature = ckpt.get("temperature", 1.0)
        best_auc    = ckpt.get("best_auc", 0.0)
        ir          = ckpt.get("instability_ratio", {})

        print(f"  Best AUC={best_auc:.4f}  Youden-t={youden_thr:.4f}  T={temperature:.3f}")
        if ir:
            print(f"  Instability ratio: {ir.get('point',0):.2f}x  "
                  f"[{ir.get('ci_lo',0):.2f}x - {ir.get('ci_hi',0):.2f}x]")

        self.stabilizer = _SimpleStabilizer(youden_thr, temperature)
        self.smoother   = _EMAsmoother(alpha=alpha, threshold=youden_thr, window=window)

    def reset(self): self.smoother.reset()

    @torch.no_grad()
    def _infer_chunk(self, audio_chunk):
        feat  = audio_to_logmel(audio_chunk)
        logit = self.model(torch.from_numpy(feat).to(self.device)).item()
        return float(sigmoid_fn(logit)), logit

    def process_chunk(self, audio_chunk):
        "Process a 1-second audio chunk. Returns result dict."
        raw_score, raw_logit     = self._infer_chunk(audio_chunk)
        chunk_dec, confidence    = self.stabilizer.predict(raw_score, raw_logit)
        smooth_dec, smooth_score = self.smoother.update(confidence)
        result = {
            "label":          "FAKE" if smooth_dec == 1 else "REAL",
            "decision":        smooth_dec,
            "confidence":      confidence,
            "raw_score":       raw_score,
            "smoothed_score":  smooth_score,
            "is_warmed_up":    self.smoother.is_warmed_up,
            "chunk_n":         self.smoother._n,
        }
        if self.verbose:
            wu  = "[warmed]" if result["is_warmed_up"] else "[------]"
            bar = chr(9608)*int(smooth_score*30) + chr(9617)*(30-int(smooth_score*30))
            col = "\033[91m" if smooth_dec==1 else "\033[92m"; rst="\033[0m"
            print(f"  Chunk {result['chunk_n']:3d}  {col}[{bar}]{rst}  "
                  f"raw={raw_score:.3f}  smooth={smooth_score:.3f}  "
                  f"{col}{result['label']}{rst}  {wu}")
        return result

    def process_file(self, file_path):
        "Process an audio file in 1-second chunks. Returns call verdict dict."
        self.reset()
        print(f"\nProcessing: {file_path}")
        y_audio, _ = librosa.load(file_path, sr=SR, mono=True)
        n_chunks   = max(1, len(y_audio) // CHUNK_SAMPLES)
        results    = [self.process_chunk(y_audio[i*CHUNK_SAMPLES:(i+1)*CHUNK_SAMPLES])
                      for i in range(n_chunks)]
        warmed     = [r for r in results if r["is_warmed_up"]] or results
        n_fake     = sum(1 for r in warmed if r["decision"]==1)
        fake_ratio = n_fake / len(warmed)
        verdict    = "FAKE" if fake_ratio >= 0.40 else "REAL"
        print(f"\n  Final verdict: {verdict}  "
              f"(fake chunks: {n_fake}/{len(warmed)} = {fake_ratio:.0%})")
        print(f"  Mean smooth score: {np.mean([r['smoothed_score'] for r in warmed]):.3f}  "
              f"Max raw score: {max(r['raw_score'] for r in warmed):.3f}")
        return {"file": file_path, "verdict": verdict, "fake_ratio": fake_ratio,
                "n_chunks": len(results),
                "mean_score": float(np.mean([r["smoothed_score"] for r in warmed])),
                "max_score":  float(np.max( [r["raw_score"]      for r in warmed]))}


print("DeepfakeCallDetector defined.")


### 4.4 Batch folder evaluation

In [ ]:
def run_folder_mode(detector, input_dir, output_csv="results.csv"):
    "Batch-evaluate all audio files in input_dir and save verdicts to CSV."
    audio_exts = {".wav",".flac",".mp3",".ogg",".m4a"}
    files = sorted([p for p in Path(input_dir).rglob("*") if p.suffix.lower() in audio_exts])
    if not files:
        print(f"No audio files found in {input_dir}"); return

    print(f"Batch evaluation: {len(files)} files -> {output_csv}")
    rows = [detector.process_file(str(f)) for f in files]

    with open(output_csv, "w", newline="") as csvf:
        writer = csv.DictWriter(csvf, fieldnames=rows[0].keys())
        writer.writeheader(); writer.writerows(rows)

    n_fake = sum(1 for r in rows if r["verdict"]=="FAKE")
    print(f"Done: {n_fake}/{len(rows)} files flagged as FAKE")
    print(f"Results saved -> {output_csv}")


print("run_folder_mode() defined.")


### 4.5 Run inference

Set `INFERENCE_MODE` and the relevant path, then execute this cell.

```
INFERENCE_MODE = "skip"    # do nothing (default — no checkpoint yet)
INFERENCE_MODE = "file"    # process a single audio file
INFERENCE_MODE = "folder"  # batch evaluate a directory
```


In [ ]:
INFERENCE_MODE  = "skip"   # change to "file" or "folder" once checkpoint exists

CHECKPOINT_PATH = str(Path(OUTPUT_DIR_TRAIN) / "deepfake_cnnlstm.pth")
INPUT_FILE      = "path/to/your/test_audio.wav"   # used when INFERENCE_MODE="file"
INPUT_DIR       = "path/to/audio/folder"           # used when INFERENCE_MODE="folder"
OUTPUT_CSV      = "results.csv"

if INFERENCE_MODE == "skip":
    print("Inference skipped. Set INFERENCE_MODE='file' or 'folder' and re-run.")

elif INFERENCE_MODE == "file":
    if not Path(CHECKPOINT_PATH).exists():
        print(f"Checkpoint not found: {CHECKPOINT_PATH} -- run Stage 2 first.")
    else:
        detector = DeepfakeCallDetector(
            CHECKPOINT_PATH, device=device, window=5, alpha=0.35, verbose=True)
        result = detector.process_file(INPUT_FILE)
        print("\nResult:", result)

elif INFERENCE_MODE == "folder":
    if not Path(CHECKPOINT_PATH).exists():
        print(f"Checkpoint not found: {CHECKPOINT_PATH} -- run Stage 2 first.")
    else:
        detector = DeepfakeCallDetector(
            CHECKPOINT_PATH, device=device, window=5, alpha=0.35, verbose=False)
        run_folder_mode(detector, INPUT_DIR, OUTPUT_CSV)


---
## Pipeline summary

| Stage | Status | Key output |
|-------|--------|------------|
| 1. Dataset preparation | Run once | `data/X_features.npy`, `y_labels.npy`, `speaker_ids.npy` |
| 2. CNN+LSTM training | Set `LOAD_CACHE=True` after first run | `outputs/deepfake_cnnlstm.pth`, 5 PDF figures |
| 3. Threshold stabilizer | Module always available | `ThresholdStabilizer`, `StabilizedCallDetector` |
| 4. Real-time inference | Set `INFERENCE_MODE` | Per-file verdicts, `results.csv` |

### Key findings reproduced by this notebook

- **AUC is stable** across speaker-independent folds — the model is a strong discriminator
- **Youden and EER thresholds vary substantially** — the instability ratio quantifies how
  much more thresholds vary than AUC, revealing deployment risk hidden by AUC-only reporting
- **Temporal EMA smoothing** prevents noisy single chunks from triggering false alarms
- **Majority-vote stabilization** across Youden + Platt + Temperature reduces operating-point drift

### Cite this work
*"On the Instability of Decision Thresholds in Deepfake Audio Detection"*,
Ahrnish P Dahal, Saurav Kumar, Anjani Kumar Verma, CIC, University of Delhi.
